# Legal Clause Simplification using LLMs

## Objective
Convert complex legal clauses into plain, easy-to-understand English.

In this notebook:
- We use an LLM (API-based) for text simplification
- Generate human-readable explanations
- Prepare outputs for final pipeline & Streamlit


In [38]:
!pip install -q pandas groq
import pandas as pd
import os


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: C:\Users\ADITYA\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
# Load output from risk analysis step
df = pd.read_csv("../outputs/clauses_with_risk_scores.csv")

df.head()


,file_name,clause,pages,class_id,label,start_at,end_at,clean_text,sentences,word_count,clause_text,risk_score,risk_level,predicted_clause_type
0,EuromediaHoldingsCorp_20070215_10SB12G_EX-10.B...,In the event that Licensor grants to another V...,2,8,Most Favored Nation,2558,2929,in the event that licensor grants to another v...,['in the event that licensor grants to another...,60,in the event that licensor grants to another v...,1,Low Risk,License Grant
1,EuromediaHoldingsCorp_20070215_10SB12G_EX-10.B...,"If Licensor enters, or has entered, into an ag...",8,8,Most Favored Nation,18515,19562,if licensor enters or has entered into an agre...,['if licensor enters or has entered into an ag...,169,if licensor enters or has entered into an agre...,3,Medium Risk,License Grant
2,EuromediaHoldingsCorp_20070215_10SB12G_EX-10.B...,"Licensor shall provide to Rogers, no later tha...",8,8,Most Favored Nation,19563,20059,licensor shall provide to rogers no later than...,['licensor shall provide to rogers no later th...,79,licensor shall provide to rogers no later than...,2,Low Risk,Audit Rights
3,IntegrityMediaInc_20010329_10-K405_EX-10.17_23...,"If for any reason, Integrity and TL are subjec...",3,8,Most Favored Nation,8173,8345,if for any reason integrity and tl are subject...,['if for any reason integrity and tl are subje...,30,if for any reason integrity and tl are subject...,0,Low Risk,License Grant
4,TomOnlineInc_20060501_20-F_EX-4.46_749700_EX-4...,"The Company will, and Online BVI will cause th...",10,8,Most Favored Nation,30765,31577,the company will and online bvi will cause the...,['the company will and online bvi will cause t...,123,the company will and online bvi will cause the...,1,Low Risk,License Grant


In [50]:
# Set up Groq API
from groq import Groq

os.environ["GROQ_API_KEY"] = "YOUR_GROQ_API_KEY_HERE"
client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

In [51]:
def build_prompt(clause_text):
    prompt = f"""
You are a legal assistant.
Simplify the following legal clause into plain English.
Avoid legal jargon. Be clear and concise.

Legal Clause:
\"\"\"{clause_text}\"\"\"

Simplified Explanation:
"""
    return prompt


In [52]:
def simplify_clause(clause_text):
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",  # Fast and powerful free model
        messages=[
            {"role": "user", "content": build_prompt(clause_text)}
        ],
        temperature=0.3,
        max_tokens=150
    )
    
    return response.choices[0].message.content.strip()

In [53]:
sample_clause = df.iloc[0]["clause_text"]

print("ORIGINAL CLAUSE:\n", sample_clause)
print("\nSIMPLIFIED VERSION:\n")
print(simplify_clause(sample_clause))


ORIGINAL CLAUSE:
 in the event that licensor grants to another vod or payperview ppv service provider in the us or the territory the right to distribute or exhibit any licensed program on an earlier availability date then licensor shall also grant to rogers the right to distribute and exhibit such licensed program on such earlier availability date on the terms provided herein

SIMPLIFIED VERSION:

Here's a simplified explanation:

"If the licensor (the company owning the content) allows another company to show a program earlier than planned, they must also give Rogers the same opportunity to show it at the same time, under the same conditions."


In [54]:
# Simplify only first 20 clauses for demo
df_sample = df.head(20).copy()

df_sample["simplified_text"] = df_sample["clause_text"].apply(simplify_clause)

df_sample[["clause_text", "simplified_text"]].head()


,clause_text,simplified_text
0,in the event that licensor grants to another v...,"Here's a simplified explanation:\n\n""If the li..."
1,if licensor enters or has entered into an agre...,Here's a simplified explanation:\n\nIf the lic...
2,licensor shall provide to rogers no later than...,Here's a simplified explanation:\n\nThe compan...
3,if for any reason integrity and tl are subject...,"Here's a simplified explanation:\n\n""If someon..."
4,the company will and online bvi will cause the...,Here's a simplified explanation:\n\nThe compan...


In [55]:
# First, generate simplified text for the sample
df_sample = df.head(20).copy()
df_sample["simplified_text"] = df_sample["clause_text"].apply(simplify_clause)

# Now create the final view
final_view = df_sample[
    ["clause_text", "predicted_clause_type", "risk_level", "simplified_text"]
]

final_view.head()


,clause_text,predicted_clause_type,risk_level,simplified_text
0,in the event that licensor grants to another v...,License Grant,Low Risk,"Here's a simplified explanation:\n\n""If the li..."
1,if licensor enters or has entered into an agre...,License Grant,Medium Risk,"Here's a simplified explanation:\n\n""If the pe..."
2,licensor shall provide to rogers no later than...,Audit Rights,Low Risk,Here's a simplified explanation:\n\nThe compan...
3,if for any reason integrity and tl are subject...,License Grant,Low Risk,"Here's a simplified explanation:\n\n""If someon..."
4,the company will and online bvi will cause the...,License Grant,Low Risk,Here's a simplified explanation of the legal c...


In [56]:
import os

os.makedirs("../outputs", exist_ok=True)

final_view.to_csv(
    "../outputs/simplified_clauses.csv",
    index=False
)

print("Simplified clauses saved to outputs/simplified_clauses.csv")


Simplified clauses saved to outputs/simplified_clauses.csv


## Summary

In this notebook, we:
- Integrated an LLM to simplify legal clauses
- Designed prompt templates for consistent outputs
- Combined clause type, risk level, and simplified explanation
- Generated human-readable legal insights

Next notebook:
➡ 06_final_pipeline.ipynb
